In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/imdb-movie-reviews/IMDB Dataset.csv


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vishakhdapat/imdb-movie-reviews")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/imdb-movie-reviews


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
import pandas as pd
df=pd.read_csv('/kaggle/input/imdb-movie-reviews/IMDB Dataset.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [3]:
df['review'] = df['review'].str.replace('<br />', '', regex=False)

In [7]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torch
import pandas as pd



# Encode labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['sentiment'])

# Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['review'].tolist(), df['label'].tolist(), test_size=0.1, random_state=42)

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Dataset class
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Create datasets
train_dataset = ReviewDataset(train_texts, train_labels, tokenizer)
val_dataset = ReviewDataset(val_texts, val_labels, tokenizer)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(label_encoder.classes_))
model.to(device)

# Optimizer & scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = len(train_loader) * num_epochs
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# Training loop
model.train()
for epoch in range(num_epochs):
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        
        loop.set_postfix(loss=loss.item())

# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)
        correct += (predictions == batch['labels']).sum().item()
        total += batch['labels'].size(0)

accuracy = correct / total
print(f"Validation Accuracy: {accuracy:.4f}")

# Save the model
model.save_pretrained("bert-finetuned-sentiment")
tokenizer.save_pretrained("bert-finetuned-sentiment")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 3/3: 100%|██████████| 2813/2813 [1:19:00<00:00,  1.69s/it, loss=0.00229]


Validation Accuracy: 0.9446


('bert-finetuned-sentiment/tokenizer_config.json',
 'bert-finetuned-sentiment/special_tokens_map.json',
 'bert-finetuned-sentiment/vocab.txt',
 'bert-finetuned-sentiment/added_tokens.json')

In [4]:
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import torch
import pandas as pd



# Encode labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['sentiment'])

# Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['review'].tolist(), df['label'].tolist(), test_size=0.1, random_state=42)

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Dataset class
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Create datasets
train_dataset = ReviewDataset(train_texts, train_labels, tokenizer)
val_dataset = ReviewDataset(val_texts, val_labels, tokenizer)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(label_encoder.classes_))
model.to(device)

# Optimizer & scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = len(train_loader) * num_epochs
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# Training loop
model.train()
for epoch in range(num_epochs):
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        
        loop.set_postfix(loss=loss.item())

# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)
        correct += (predictions == batch['labels']).sum().item()
        total += batch['labels'].size(0)

accuracy = correct / total
print(f"Validation Accuracy: {accuracy:.4f}")

# Save the model
model.save_pretrained("bert-finetuned-sentiment")
tokenizer.save_pretrained("bert-finetuned-sentiment")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 3/3: 100%|██████████| 2813/2813 [1:20:06<00:00,  1.71s/it, loss=0.00248]


Validation Accuracy: 0.9484


('bert-finetuned-sentiment/tokenizer_config.json',
 'bert-finetuned-sentiment/special_tokens_map.json',
 'bert-finetuned-sentiment/vocab.txt',
 'bert-finetuned-sentiment/added_tokens.json')

In [5]:
import shutil
shutil.make_archive('bert-finetuned-sentiment', 'zip', 'bert-finetuned-sentiment')


'/kaggle/working/bert-finetuned-sentiment.zip'

In [6]:
import shutil
shutil.make_archive('/kaggle/working/bert-finetuned-sentiment', 'zip', '/kaggle/working/bert-finetuned-sentiment')


'/kaggle/working/bert-finetuned-sentiment.zip'

In [7]:
from IPython.display import FileLink

# Make sure the zip was created successfully
import shutil
shutil.make_archive('/kaggle/working/bert-finetuned-sentiment1', 'zip', '/kaggle/working/bert-finetuned-sentiment1')

# Create a clickable link
FileLink(r'/kaggle/working/bert-finetuned-sentiment.zip')


/kaggle/working/bert-finetuned-sentiment.zip

In [20]:
from transformers import BertForSequenceClassification, BertTokenizer
import torch

# Load from saved folder
model = BertForSequenceClassification.from_pretrained("bert-finetuned-sentiment")
tokenizer = BertTokenizer.from_pretrained("bert-finetuned-sentiment")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

label_map = {0: "negative", 1: "positive"}  # Or your actual class mapping

def predict_sentiment(model, tokenizer, texts, device):
    model.eval()
    encoded = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**encoded)
        predictions = torch.argmax(outputs.logits, dim=-1)
    
    return [label_map[pred.item()] for pred in predictions]

random_reviews = [
    "I should have known when I heard Anne Rice left the project that the movie would disappoint me. I couldn't have predicted that years after it's release just thinking about the movie still makes me angry. The novels are amazing, and while I understand much gets lost in the translation to screen, this movie was a great big middle finger to her original work. I hope one day someone tries again, the right way, starting with The Vampire Lestat. They change the roles and looks of major and minor characters alike for no good reason. They destroy Lestat's history. The acting of the Queen is exaggerated to the point of comedy, but I just can't bring myself to laugh. The charm and allure of the novels just isn't there. The movie is a bad excuse to cram as many musicians and dark imagery as possible into one movie, hoping the teeny Goths of America would lap it up. Part of the appeal of the first movie, of Louis story, is that he is caught between his humanity and his curse. Lestat is supposed to take over and display the magic and excitement of the vampire world. Thank goodness I read the books first, or I'd have never touched them after this movie."
    "I didn't like the film at all. Waste of time.",
    "It was okay. There is always a scope to get better"
]

results = predict_sentiment(model, tokenizer, random_reviews, device)
for review, sentiment in zip(random_reviews, results):
    print(f"Review: \"{review}\" => Sentiment: {sentiment}")


Review: "I should have known when I heard Anne Rice left the project that the movie would disappoint me. I couldn't have predicted that years after it's release just thinking about the movie still makes me angry. The novels are amazing, and while I understand much gets lost in the translation to screen, this movie was a great big middle finger to her original work. I hope one day someone tries again, the right way, starting with The Vampire Lestat. They change the roles and looks of major and minor characters alike for no good reason. They destroy Lestat's history. The acting of the Queen is exaggerated to the point of comedy, but I just can't bring myself to laugh. The charm and allure of the novels just isn't there. The movie is a bad excuse to cram as many musicians and dark imagery as possible into one movie, hoping the teeny Goths of America would lap it up. Part of the appeal of the first movie, of Louis story, is that he is caught between his humanity and his curse. Lestat is su

In [ ]:
# from IPython.display import FileLink 
# FileLink('bert-finetuned-sentiment')